<a href="https://colab.research.google.com/github/Hyounseo/Opensource-FISH/blob/main/%EC%98%A4%EC%86%8C%ED%94%84%ED%8C%80%ED%94%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 32.4 MB/s eta 0:00:00


In [13]:
import easyocr
import os
import pandas as pd

reader = easyocr.Reader(['ko'])

data = []

folders = {
    '/content/drive/MyDrive/AI_Phishing_Project': 1,
    '/content/drive/MyDrive/AI_OpenSourceProgramming_Correct ': 0
}

for folder_path, label in folders.items():
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):  # lower() 추가!
            full_path = os.path.join(folder_path, fname)
            result = reader.readtext(full_path, detail=0)
            text = ' '.join(result)
            data.append({'filename': fname, 'text': text, 'label': label})
            print(f'완료: {fname} → {text[:30]}...')

df = pd.DataFrame(data)
df.to_csv('/content/drive/MyDrive/dataset.csv', index=False)
print(f'\n총 {len(df)}개 처리 완료!')
print(df['label'].value_counts())

완료: 스크린샷 2026-05-13 111437.png → [민원24]쓰레기 무단투기로 단속되어 과태료 부과되엿습...
완료: test.png → 0+ 1038 0 8  10096 054-472-704...
완료: 스크린샷 2026-05-29 165241.png → 못데리아" 한우불고기세트 사용구품 2매 "++명:/_ ...
완료: KakaoTalk_20260529_165046168_02.png → 4:43 너긋 161 007773117880683 문자...
완료: KakaoTalk_20260529_165046168_03.jpg → 4:43 (// -긋 161 00777311742409...
완료: 스크린샷 2026-05-29 165407.png → 11월 26일 일요일 아버님께서 금일아침에 별세하세기에...
완료: KakaoTalk_20260529_170029744_02.png → 4:56 너긋 153 +82 10-7905-5162 문...
완료: KakaoTalk_20260529_170029744_01.png → 4:56 너긋 153 +82 10-3098-5979 문...
완료: KakaoTalk_20260529_170029744.png → 4:56 너긋 153 +82 10-3098-5979 문...
완료: KakaoTalk_20260529_170029744_04.png → 4:56 너긋 153 +82 10-5734-4815 문...
완료: KakaoTalk_20260529_170029744_05.png → 4:55 ( 너긋 153 +82 10-4703-5840...
완료: KakaoTalk_20260529_170029744_03.png → 4:56 너긋 153 +82 10-6611-4013 [...
완료: Screenshot_20260529_170231_Messages.jpg → -*7 5.02 [;1 @0 "1 18 02-790-0...
완료: Screenshot_20260529_170212_Messages.jpg → -

In [14]:
print(df['label'].value_counts())

label
1    93
0    69
Name: count, dtype: int64


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import pickle

df = pd.read_csv('/content/drive/MyDrive/dataset.csv')
df = df.dropna(subset=['text'])

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(f'정확도: {accuracy_score(y_test, y_pred):.2f}')
print(classification_report(y_test, y_pred))

# 모델 저장
with open('/content/drive/MyDrive/model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('/content/drive/MyDrive/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('모델 저장 완료!')

정확도: 0.91
              precision    recall  f1-score   support

           0       1.00      0.80      0.89        15
           1       0.86      1.00      0.92        18

    accuracy                           0.91        33
   macro avg       0.93      0.90      0.91        33
weighted avg       0.92      0.91      0.91        33

모델 저장 완료!


In [16]:
!pip install gradio

In [17]:
import gradio as gr
import pickle
import easyocr
from PIL import Image
import numpy as np

# 저장된 모델 불러오기
with open('/content/drive/MyDrive/model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('/content/drive/MyDrive/vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

reader = easyocr.Reader(['ko'])

def predict(image):
    # 이미지에서 텍스트 추출
    img_array = np.array(image)
    result = reader.readtext(img_array, detail=0)
    text = ' '.join(result)

    if not text.strip():
        return "텍스트를 인식하지 못했습니다.", "알 수 없음", 0

    # 피싱 확률 계산
    vec = vectorizer.transform([text])
    proba = model.predict_proba(vec)[0]
    phishing_prob = proba[1] * 100

    if phishing_prob >= 50:
        result_text = f"⚠️ 피싱 의심 문자입니다!"
        label = "피싱"
    else:
        result_text = f"✅ 정상 문자입니다."
        label = "정상"

    return result_text, text, round(phishing_prob, 2)

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type='pil', label='문자 캡처 이미지 업로드'),
    outputs=[
        gr.Textbox(label='판별 결과'),
        gr.Textbox(label='추출된 텍스트'),
        gr.Number(label='피싱 위험도 (%)'),
    ],
    title='🎣 AI 피싱 탐지기',
    description='문자 캡처 이미지를 업로드하면 피싱 여부를 판별해드립니다.'
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f13ddc585111a6807a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
